In [2]:
%load_ext autoreload
%autoreload 2
from utils.plotting import *

In [6]:
dir_path_seeds = np.array([
    [
"20260327-051120_MLP_FSNet_seed0_nepochs300_lr0.0001_trainsize7000_finetune_20260327-050238_sup_seedpen_model_990",
"20260327-052711_MLP_FSNet_seed3_nepochs300_lr0.0001_trainsize7000_finetune_20260327-050238_sup_seedpen_model_990",
"20260327-052720_MLP_FSNet_seed1_nepochs300_lr0.0001_trainsize7000_finetune_20260327-050238_sup_seedpen_model_990",
"20260327-051121_MLP_FSNet_seed2_nepochs300_lr0.0001_trainsize7000_finetune_20260327-050238_sup_seedpen_model_990",
    ]
])

In [4]:
# getting results
rel_path = "./results/nonsmooth_nonconvex/socp/SOCPProblem-100-50-50-10000"

# collecting data
obj_mean = np.zeros_like(dir_path_seeds, dtype=float)
obj_max = np.zeros_like(dir_path_seeds, dtype=float)
eq_violation_l1_mean = np.zeros_like(dir_path_seeds, dtype=float)
eq_violation_l1_max = np.zeros_like(dir_path_seeds, dtype=float)
ineq_violation_l1_mean = np.zeros_like(dir_path_seeds, dtype=float)
ineq_violation_l1_max = np.zeros_like(dir_path_seeds, dtype=float)
merit_mean = np.zeros_like(dir_path_seeds, dtype=float)
merit_max = np.zeros_like(dir_path_seeds, dtype=float)

batch_size = 256
for i in range(dir_path_seeds.shape[0]):  # over ckpt
    for j in range(dir_path_seeds.shape[1]):  # over seeds
        dir_path = os.path.join(
            rel_path, dir_path_seeds[i, j]
        )
        with open(os.path.join(dir_path, "results.pkl"), "rb") as f:
            print(dir_path)
            results = pickle.load(f)
            results_ = results['test_results']['batch_size_comparison'][batch_size]['metrics']
        obj_mean[i, j] = results_["objective"] 
        obj_max[i, j] = results_["objective_max"] 
        eq_violation_l1_mean[i, j] = results_["eq_violation_l1_mean"]
        eq_violation_l1_max[i, j] = results_["eq_violation_l1_max"]
        ineq_violation_l1_mean[i, j] = results_["ineq_violation_l1_mean"]
        ineq_violation_l1_max[i, j] = results_["ineq_violation_l1_max"]
        merit_mean[i, j] = results_["merit_mean"]
        merit_max[i, j] = results_["merit_max"]
        
# assuming you already have: dir_path_seeds, rel_path
num_baselines = dir_path_seeds.shape[0]
num_seeds = dir_path_seeds.shape[1]
print(num_baselines, num_seeds)

# read one file to know number of epochs
with open(os.path.join(rel_path, dir_path_seeds[0, 0], "results.pkl"), "rb") as f:
    results = pickle.load(f)
    num_epochs = len(results["val_history"])

# store [num_baselines, num_seeds, num_epochs]
obj_mean_epochs = np.zeros((num_baselines, num_seeds, num_epochs))

for i in range(num_baselines):  # over ckpt
    for j in range(num_seeds):  # over seed
        dir_path = os.path.join(rel_path, dir_path_seeds[i, j])
        with open(os.path.join(dir_path, "results.pkl"), "rb") as f:
            results = pickle.load(f)

./results/nonsmooth_nonconvex/socp/SOCPProblem-100-50-50-10000/20260327-050313_MLP_FSNet_seed2_nepochs300_lr0.0001_trainsize7000_finetune_20260327-045910_sup_seedpen_model_990
./results/nonsmooth_nonconvex/socp/SOCPProblem-100-50-50-10000/20260327-051848_MLP_FSNet_seed3_nepochs300_lr0.0001_trainsize7000_finetune_20260327-045910_sup_seedpen_model_990
./results/nonsmooth_nonconvex/socp/SOCPProblem-100-50-50-10000/20260327-050413_MLP_FSNet_seed0_nepochs300_lr0.0001_trainsize7000_finetune_20260327-045910_sup_seedpen_model_990
./results/nonsmooth_nonconvex/socp/SOCPProblem-100-50-50-10000/20260327-052002_MLP_FSNet_seed1_nepochs300_lr0.0001_trainsize7000_finetune_20260327-045910_sup_seedpen_model_990
./results/nonsmooth_nonconvex/socp/SOCPProblem-100-50-50-10000/20260327-051120_MLP_FSNet_seed0_nepochs300_lr0.0001_trainsize7000_finetune_20260327-050238_sup_seedpen_model_990
./results/nonsmooth_nonconvex/socp/SOCPProblem-100-50-50-10000/20260327-052711_MLP_FSNet_seed3_nepochs300_lr0.0001_train

In [5]:
# Define metrics and headers
metrics = [
    ("Obj Mean", obj_mean),
    # ("Opt Gap Max", obj_max),
    ("Eq Vio Mean", eq_violation_l1_mean),
    ("Eq Vio Max", eq_violation_l1_max),
    ("Ineq Vio Mean", ineq_violation_l1_mean),
    ("Ineq Vio Max", ineq_violation_l1_max),
    # ("Merit Mean", merit_mean),
    # ("Merit Max", merit_max),
]

# Print Header
header = f"{'Method':<8} | " + " | ".join([f"{name:<18}" for name, _ in metrics])
print(header)
print("-" * len(header))

# Print Rows (Method)
for i in range(num_baselines):
    row_str = f"{i:<8} | "
    for _, data in metrics:
        # Compute mean and std over seeds (axis 1)
        mu = np.mean(data[i])
        sigma = np.std(data[i])
        # Format as scientific notation
        if data is obj_mean or data is obj_max:
            row_str += f"{mu:.2f} ± {sigma:.2f}".ljust(18) + " | "
        else:
            row_str += f"{mu:.2e} ± {sigma:.2e}".ljust(18) + " | "
    print(row_str)

Method   | Obj Mean           | Eq Vio Mean        | Eq Vio Max         | Ineq Vio Mean      | Ineq Vio Max      
-----------------------------------------------------------------------------------------------------------------
0        | -3.72 ± 0.03       | 6.60e-05 ± 4.53e-06 | 1.19e-03 ± 1.71e-04 | 4.47e-07 ± 7.65e-08 | 2.91e-05 ± 2.09e-06 | 
1        | 1.46 ± 0.68        | 5.15e-05 ± 2.39e-05 | 5.00e-03 ± 5.52e-03 | 3.78e-06 ± 5.51e-06 | 9.06e-04 ± 1.42e-03 | 
